In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time

In [ ]:
transform = transforms.ToTensor()

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 784)       # flatten 28x28 image to 784
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = SimpleNet()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model(images)
        loss = loss_fn(output, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} done")

torch.save(model.state_dict(), 'model_fp32.pth')
print("Model saved")

In [ ]:
def evaluate(model, test_loader, dtype=None):
    model.eval()
    correct = 0
    total = 0

    # measure memory
    param_bytes = sum(p.nbytes for p in model.parameters())

    # warm up run
    for images, labels in test_loader:
        if dtype:
            images = images.to(dtype)
        with torch.no_grad():
            model(images)
        break

    # timed run
    start = time.perf_counter()
    with torch.no_grad():
        for images, labels in test_loader:
            if dtype:
                images = images.to(dtype)
            output = model(images)
            preds = output.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    elapsed = time.perf_counter() - start

    accuracy = correct / total * 100
    return accuracy, param_bytes, elapsed

In [ ]:
import copy

results = {}

# FP32 baseline
model_fp32 = SimpleNet()
model_fp32.load_state_dict(torch.load('model_fp32.pth'))
acc, mem, lat = evaluate(model_fp32, test_loader)
results['FP32'] = (acc, mem, lat)
print(f"FP32  — Acc: {acc:.2f}%  Mem: {mem} bytes  Latency: {lat:.4f}s")

# FP16
model_fp16 = copy.deepcopy(model_fp32).half()
acc, mem, lat = evaluate(model_fp16, test_loader, dtype=torch.float16)
results['FP16'] = (acc, mem, lat)
print(f"FP16  — Acc: {acc:.2f}%  Mem: {mem} bytes  Latency: {lat:.4f}s")

# BF16
model_bf16 = copy.deepcopy(model_fp32).to(torch.bfloat16)
acc, mem, lat = evaluate(model_bf16, test_loader, dtype=torch.bfloat16)
results['BF16'] = (acc, mem, lat)
print(f"BF16  — Acc: {acc:.2f}%  Mem: {mem} bytes  Latency: {lat:.4f}s")

# INT8
torch.backends.quantized.engine = 'qnnpack'
model_int8 = torch.quantization.quantize_dynamic(
    copy.deepcopy(model_fp32),
    {nn.Linear},
    dtype=torch.qint8
)
acc, mem, lat = evaluate(model_int8, test_loader)
results['INT8'] = (acc, mem, lat)
print(f"INT8  — Acc: {acc:.2f}%  Mem: {mem} bytes  Latency: {lat:.4f}s")

In [ ]:
import matplotlib.pyplot as plt

formats = list(results.keys())
accs = [results[f][0] for f in formats]
mems = [results[f][1] / 1024 for f in formats]  # KB
lats = [results[f][2] for f in formats]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(formats, accs)
axes[0].set_title('Accuracy (%)')
axes[0].set_ylim(90, 100)

axes[1].bar(formats, mems)
axes[1].set_title('Memory (KB)')

axes[2].bar(formats, lats)
axes[2].set_title('Latency (s)')

plt.tight_layout()
plt.savefig('results.png')
plt.show()